In [25]:
import pandas as pd
from sqlalchemy import create_engine,URL,text

In [26]:
def data_extraction():
    try:
        connection_url = URL.create(
            drivername="mysql+pymysql",
            username="root",
            password="Shantanu@2026",
            host="localhost",
            port=3306,
            database="sales_analytics"
        )

        engine = create_engine(connection_url)

        sales = pd.read_sql("SELECT * FROM sales_source", engine)

        return sales

    except Exception as e:
        print(f"Error: {e}")

In [27]:

def data_transformation(df):

    df = df.drop_duplicates()

    # Remove records with missing identifiers
    df = df[df["sale_id"].notna()]
    df = df[df["order_id"].notna()]
    df = df[df["customer_id"].notna()]
    df = df[df["product_id"].notna()]

    # Quantity transformation
    df["quantity"] = df["quantity"].fillna(0)
    df.loc[df["quantity"] <= 0, "quantity"] = 1

    # Unit price transformation
    avg_price = df.loc[df["unit_price"] > 0, "unit_price"].mean()

    df["unit_price"] = df["unit_price"].fillna(avg_price)
    df.loc[df["unit_price"] <= 0, "unit_price"] = avg_price

    # Total amount transformation
    calculated_amount = df["unit_price"] * df["quantity"]

    df["total_amount"] = df["total_amount"].fillna(calculated_amount)

    #Date
    df['sale_date'] = pd.to_datetime(df['sale_date'])
    df['year'] = df['sale_date'].dt.year
    df['month'] = df['sale_date'].dt.month
    df['month_name'] = df['sale_date'].dt.month_name()
    df['quarter'] = df['sale_date'].dt.quarter

    return df


In [28]:
df = data_extraction()
transforemed_df = data_transformation(df)

In [29]:
transforemed_df.head()

,sale_id,order_id,customer_id,product_id,sale_date,quantity,unit_price,total_amount,year,month,month_name,quarter
0,1,1001,101,501,2024-01-15,2,500.0,1000.0,2024,1,January,1
1,2,1002,102,502,2024-01-25,3,200.0,600.0,2024,1,January,1
2,3,1003,103,503,2024-02-10,1,1500.0,1500.0,2024,2,February,1
3,4,1004,101,504,2024-03-20,4,250.0,1000.0,2024,3,March,1
4,5,1005,104,501,2024-04-12,2,500.0,1000.0,2024,4,April,2


In [ ]:
# year 2024 report
year_report = transforemed_df.groupby('year')['total_amount'].sum()
month_report_2024 = transforemed_df[transforemed_df["year"]==2024].groupby('month')['total_amount'].sum()
quarter_report_2024 = transforemed_df[transforemed_df["year"]==2024].groupby('quarter')['total_amount'].sum()

In [36]:
# year 2025 report
year_given = 2025
yearly_report = transforemed_df.groupby("year")["total_amount"].sum()
monthly_report = transforemed_df[transforemed_df["year"]==year_given].groupby("month")["total_amount"].sum()
quarterly_report = transforemed_df[transforemed_df["year"]==year_given].groupby("quarter")["total_amount"].sum()

In [39]:
yearly_report

year
2024    15650.0
2025    18750.0
2026    10400.0
Name: total_amount, dtype: float64

In [37]:
monthly_report

month
1     1500.0
2     1000.0
3     3000.0
4     1000.0
5     1000.0
6     1200.0
7     1500.0
8     2000.0
9     1500.0
10     800.0
11    3000.0
12    1250.0
Name: total_amount, dtype: float64

In [38]:
quarterly_report

quarter
1    5500.0
2    3200.0
3    5000.0
4    5050.0
Name: total_amount, dtype: float64